# MLB Free Agent Contract Prediction Pipeline — v3

#### Enhancements over v2:
  1. Team-interest features from MLBTR scraper (mlb_scraper_ner.py)
     - n_teams_interested, n_rumor_mentions, has_big_market_interest,
       first_rumor_doy, rumor_intensity
     - Gracefully degrades: if the cache CSV doesn't exist, all
       team-interest features are filled with 0 / neutral values so
       the rest of the pipeline runs unaffected.
  2. WAR-tiered Phase 2 regression models
     - Four separate GBM/Ridge models, one per WAR tier
     - Soft sigmoid blending at tier boundaries to prevent prediction
       cliffs at the cutoff values
     - Per-tier evaluation to see exactly where improvement comes from
  3. Full visualisation suite expanded to cover tiered model comparison
     and team-interest feature distributions

## Pipeline Phases:
-------------
  Phase 1  : GBM classifier  — MLB vs. Minor League contract

  Phase 2a : Tiered GBM/Ridge regressors — AAV  (MLB contracts only)

  Phase 2b : Tiered GBM/Ridge regressors — Years (MLB contracts only)

## Needed Imports:
--------------
Aside from the libraries below, you will need to ensure you have added the following csv files:
- mlb_players.csv
- mlb_contracts.csv
- team_interest_cache.csv

In [1]:
# import needed tools and libraries

import pandas as pd
import numpy as np
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    mean_absolute_error, r2_score, mean_squared_error
)
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import joblib

warnings.filterwarnings('ignore')

## Part 1: Market / Inflation Measures

Why add market features? Free-agent salaries are not set in a vacuum. Teams bid against each other, and the *overall market* rises or falls each year based on:
- Total MLB revenue (determines how much teams can afford)
- The Competitive Balance Tax (CBT / "luxury tax") threshold — a de-facto soft cap that anchors team spending behaviour
- League-wide mean AAV in that offseason (direct peer salary benchmark)

Without these, a 4-WAR player who signed in 2016 looks identical to one who signed in 2023, even though the latter market paid ~30% more.

Sources / derivation:
- MLB revenues: publicly reported figures (MLBPA, Forbes, Statista)
- CBT thresholds: each CBA; official per year
- League avg AAV index: computed below from the contracts dataset itself, so it is always internally consistent with our training data.

In [2]:
# establish table of MLB's revenue and CBT over time

# revenue listed in billions USD
# CBT threshold listed in millions USD

MARKET_TABLE = {
    # year : (mlb_revenue_B, cbt_threshold_M)
    2016: (9.0,  189.0),
    2017: (9.5,  195.0),
    2018: (10.3, 197.0),
    2019: (10.7, 206.0),
    2020: (3.6,  208.0),   # COVID — revenue cratered
    2021: (9.0,  210.0),
    2022: (10.8, 230.0),   # new CBA reset CBT higher
    2023: (11.4, 233.0),
    2024: (11.9, 237.0),
    2025: (12.5, 241.0),
}


## Part 2: Load and Clean Datasets

In [3]:
# load datasets
contracts = pd.read_csv('/content/mlb_contracts.csv')
players   = pd.read_csv('/content/mlb_player.csv')


**Shohei** Ohtani is a two-way pitcher/hitter — every standard projection model breaks on him because his WAR combines two completely different skill profiles.

Because of this, we will remove him from both datasets so he doesn't distort feature distributions.

We also can only merge these datasets where a shared key exists, that being MLBAM ID. We will need to drop the rows that lack the MLBAM ID.

In [4]:
# remove Ohtani
contracts = contracts[~contracts['name'].str.contains('Ohtani', na=False)].copy()
players   = players[~players['Name'].str.contains('Ohtani', na=False)].copy()

# We can only merge datasets MLBAM ID is non-null; drop contracts that lack one
contracts = contracts[contracts['mlbam_id'].notna()].copy()
contracts['mlbam_id'] = contracts['mlbam_id'].astype(int)

print(f"  Contracts      : {len(contracts):,}")
print(f"  Player-seasons : {len(players):,}")

  Contracts      : 2,246
  Player-seasons : 5,129


## Part 3: Team-Interest Feature Cache

The MLBTR scraper notebook writes the CSV for this part.  If it doesn't exist yet, we fill all team-interest features with neutral defaults so the rest of the pipeline runs without interruption.

Neutral defaults:
- n_teams_interested = 0      (unknown — treated as no data)
- n_rumor_mentions   = 0
- has_big_market_interest = 0
- first_rumor_doy    = 365    (latest possible — signals no early rumours)
- rumor_intensity    = 0

Once you have the cache, simply re-run this script — no other changes needed.

In [5]:
# build list of columns for team-interest

TEAM_INTEREST_COLS = [
    'n_teams_interested',
    'n_rumor_mentions',
    'has_big_market_interest',
    'first_rumor_doy',
    'rumor_intensity',
]

# pull csv for team-interest
TEAM_INTEREST_CACHE = '/content/team_interest_cache.csv'

try:
    ti_df = pd.read_csv(TEAM_INTEREST_CACHE)
    ti_df['mlbam_id']      = ti_df['mlbam_id'].astype(int)
    ti_df['contract_year'] = ti_df['contract_year'].astype(int)
    # Merge onto contracts on (mlbam_id, contract_year)
    contracts = contracts.merge(
        ti_df[['mlbam_id', 'contract_year'] + TEAM_INTEREST_COLS],
        left_on  = ['mlbam_id', 'Contract Year'],
        right_on = ['mlbam_id', 'contract_year'],
        how='left',
    )
    n_matched = contracts[TEAM_INTEREST_COLS[0]].notna().sum()
    print(f"\n  Team-interest cache loaded: {n_matched:,} / {len(contracts):,} rows matched")
    # Fill any unmatched rows with neutral defaults
    for col in TEAM_INTEREST_COLS:
        contracts[col] = contracts[col].fillna(
            365 if col == 'first_rumor_doy' else 0
        )
    HAS_TEAM_INTEREST = True

except FileNotFoundError:
    print(
        f"\n  [INFO] Team-interest cache not found at '{TEAM_INTEREST_CACHE}'.\n"
        "         Run mlb_scraper_ner.py to generate it.\n"
        "         Proceeding with neutral defaults (0 / 365)."
    )
    for col in TEAM_INTEREST_COLS:
        contracts[col] = 365 if col == 'first_rumor_doy' else 0
    HAS_TEAM_INTEREST = False



  [INFO] Team-interest cache not found at '/content/team_interest_cache.csv'.
         Run mlb_scraper_ner.py to generate it.
         Proceeding with neutral defaults (0 / 365).


## Part 4: Compute MLB's Average AAV Index from Dataset

Computing the league average AAV per year directly from our dataset means the index always moves in lockstep with the contracts we're modeling. We only use *MLB* contracts (mlb_contract == 1) so minor-league deals don't artificially depress the baseline.

We also compute the log of the league AAV — models that predict log(AAV) benefit from a log-scale market anchor rather than a raw dollar figure.

In [6]:
# identify median MLB free agent contract AAV by year

mlb_aav_by_year = (
    contracts[contracts['mlb_contract'] == 1]
    .groupby('Contract Year')['aav']
    .median()  # median should protect against outliers
    .rename('league_median_aav')
)

# Normalise to 2019 = 1.0 so the feature is a pure inflation multiplier.
# 2019 is the last "normal" pre-COVID season and a natural reference point.
BASE_YEAR = 2019
base_aav  = mlb_aav_by_year.get(BASE_YEAR, mlb_aav_by_year.mean())
aav_index = (mlb_aav_by_year / base_aav).rename('aav_market_index')

print("\nLeague AAV index (2019 = 1.0):")
for yr, val in aav_index.items():
    print(f"  {yr}: {val:.3f}")



League AAV index (2019 = 1.0):
  2016: 1.375
  2017: 1.000
  2018: 0.938
  2019: 1.000
  2020: 0.750
  2021: 1.250
  2022: 1.562
  2023: 0.750
  2024: 0.875
  2025: 1.438


## Part 5: Identify the Performance Stat Features

These are the per-season statistics we extract from the player data.

Chosen to cover:
- Value / production  : WAR, wRC+, wOBA, wRAA, OBP+
- Plate discipline    : BB%, K%, ISO
- Contact quality     : BABIP, Barrel%, HardHit%, xwOBA, xSLG
- Counting stats      : HR, SB
- Defence   : Def
- Baserunning / Overall Offense : BsR, Off, Spd
- Usage               : PA, G
- Context             : WPA, Clutch
- Age                 : Age (needed for delta features to detect aging curves)

In [7]:
# Performance stat features to include

STAT_COLS = [
    'WAR', 'wRC+', 'wOBA', 'OBP', 'SLG', 'ISO', 'BABIP',
    'BB%', 'K%', 'HR', 'SB', 'Def', 'BsR', 'Off',
    'Barrel%', 'HardHit%', 'xwOBA', 'xSLG', 'PA', 'G',
    'wRAA', 'OBP+', 'ISO+', 'WPA', 'Clutch', 'Spd', 'Age'
]



## Part 6: Feature Engineering

Build a flat feature vector for each player and contract_year pair.

### Performance features

For each stat in STAT_COLS we compute:
- last_*     : value in the contract year itself (most recent signal)
- mean_*     : average over the lookback window  (stability signal)
- delta1_*   : change from year-1 to year-0      (short-term trajectory)
- delta2_*   : change from year-2 to year-0      (medium-term trajectory)
- trend_*    : OLS slope over all available seasons (direction of career arc)

Computing these deltas can be useful because a player posting 3 WAR who was at 1 WAR last year is on an upswing and may command more than a player posting 3 WAR who was at 5 WAR — the raw level stat alone misses this context.

Market features
- mlb_revenue_B   : total MLB revenue that year
- cbt_threshold_M : luxury-tax threshold (soft cap anchor)
- aav_market_index: league median AAV relative to 2019 baseline
- log_cbt         : log of CBT threshold — revenue features are right-skewed

Parameters
- player_df    : full players DataFrame
- contract_row : pandas Series — one row from the contracts DataFrame
- market_table : dict year -> (revenue_B, cbt_threshold_M)
- aav_index    : Series year -> market index float
- lookback     : how many seasons back to include (default 3)

Returns a dict of features, or None if no player data found


In [8]:
# build function for feature engineering

def build_player_features(player_df, contract_row, market_table, aav_index, lookback=3):

    mlbam_id      = contract_row['mlbam_id']
    contract_year = contract_row['Contract Year']

# Retrieve all seasons for this player up to and including the contract year
    hist = player_df[
        (player_df['MLBAMID'] == mlbam_id) &
        (player_df['Season']  <= contract_year) &
        (player_df['Season']  >= contract_year - lookback)
    ].sort_values('Season')

    if hist.empty:
        return None

    feats       = {}
    n_seasons   = len(hist)
    feats['n_prior_seasons'] = n_seasons # can serve as proxy for service time
    last        = hist.iloc[-1]

    # ── 6a. Last-season stats ─────────────────────────────────────────────────
    for col in STAT_COLS:
        feats[f'last_{col}'] = last[col] if (col in last.index and pd.notna(last[col])) else np.nan

    # ── 6b. Career-window means ───────────────────────────────────────────────
    for col in STAT_COLS:
        feats[f'mean_{col}'] = hist[col].mean() if col in hist.columns else np.nan

    # ── 6c. One-year delta: season[t] - season[t-1] ───────────────────────────
    if n_seasons >= 2:
        prev = hist.iloc[-2]
        for col in STAT_COLS:
            if col in hist.columns:
                feats[f'delta1_{col}'] = (
                    last[col] - prev[col]
                    if pd.notna(last[col]) and pd.notna(prev[col]) else np.nan
                )
    else:
        for col in STAT_COLS:
            feats[f'delta1_{col}'] = np.nan

    # ── 6d. Two-year delta: season[t] - season[t-2] ───────────────────────────
    if n_seasons >= 3:
        two_back = hist.iloc[-3]
        for col in STAT_COLS:
            if col in hist.columns:
                feats[f'delta2_{col}'] = (
                    last[col] - two_back[col]
                    if pd.notna(last[col]) and pd.notna(two_back[col]) else np.nan
                )
    else:
        for col in STAT_COLS:
            feats[f'delta2_{col}'] = np.nan

    # ── 6e. Linear trend slope ────────────────────────────────────────────────
    # a positive slope means the player is improving
    # a negative slope means the player is declining
    # using np.ployfit(degree = 1) to give slope & intercept
    for col in STAT_COLS:
        if col in hist.columns and hist[col].notna().sum() >= 2:
            vals = hist[col].dropna().values
            feats[f'trend_{col}'] = float(np.polyfit(np.arange(len(vals)), vals, 1)[0])
        else:
            feats[f'trend_{col}'] = np.nan

    # ── 6f. Market / inflation features ──────────────────────────────────────
    rev_B, cbt_M = market_table.get(contract_year, (10.0, 210.0))
    feats['mlb_revenue_B']    = rev_B
    feats['cbt_threshold_M']  = cbt_M
    feats['log_cbt']          = np.log(cbt_M) # helps for skew
    feats['aav_market_index'] = float(aav_index.get(contract_year, 1.0))
    feats['contract_year_num'] = contract_year

    # ── 6g. Team-interest features (from MLBTR cache) ────────────────────────
    # These are already merged onto contract_row in Part 3.
    # We just copy them into the feature dict here.
    for col in TEAM_INTEREST_COLS:
        feats[col] = contract_row.get(col, 0)

    return feats


## Part 7: Build the Overall Feature Dataset

In [9]:
print("\nEngineering features for each contract…")

# build the master feature dataset
rows = []
for _, contract_row in contracts.iterrows():
    feats = build_player_features(players, contract_row, MARKET_TABLE, aav_index)
    if feats is None:
        continue

    # attach target variables and identifiers
    feats['mlbam_id']      = contract_row['mlbam_id']
    feats['name']          = contract_row['name']
    feats['pos']           = contract_row['pos']
    feats['contract_year'] = contract_row['Contract Year']
    feats['mlb_contract']  = contract_row['mlb_contract']
    feats['aav']           = contract_row['aav']
    feats['yrs']           = contract_row['yrs']
    rows.append(feats)


df = pd.DataFrame(rows)
print(f"  Dataset: {len(df):,} rows × {df.shape[1]} columns")
print(f"  MLB contracts   : {int(df['mlb_contract'].sum()):,}")
print(f"  Minor contracts : {int((df['mlb_contract']==0).sum()):,}")

# one-hot encode player position (dropping first level to avoid multicollinearity)
df = pd.get_dummies(df, columns=['pos'], drop_first=True, dtype=float)



Engineering features for each contract…
  Dataset: 1,679 rows × 153 columns
  MLB contracts   : 543
  Minor contracts : 1,136


## Part 8: Player Clustering

Clustering is likely an important addition to our process. Free-agent markets are implicitly segmented. Scouts and GMs don't compare a power-hitting 1B to a slap-hitting CF; they compare each player to their "archetype" peers. Clustering formalises this:
- An elite power hitter (cluster A) likely commands a very different AAV curve from a glove-first defender (cluster B) even at the same WAR level.
- Adding the cluster label (and soft memberships via distances) allows the regression models to fit separate implicit salary curves per archetype, rather than averaging across all player types.
- The classifier also benefits: minor-leaguers tend to cluster together in performance space, giving the model an extra structural signal.

Approach:
- Cluster on "career profile" stats — means and trends over the lookback window.  We do NOT include target variables (aav, yrs) to avoid leakage.
- Use PCA before K-Means:
  - Removes multicollinearity between correlated stats (e.g. wRC+ and wOBA)
  - Speeds up K-Means convergence
  - Makes cluster shapes more spherical (K-Means assumes spherical clusters)
- Number of clusters: 6.  Chosen to map roughly to archetypes:
  - (1) elite hitters,
  - (2) power bats,
  - (3) average regulars,
  - (4) glove-first/utility,
  - (5) fringe MLB,
  - (6) minor-league calibre.

We validate this with inertia (elbow) and silhouette score below.

In [10]:
# import the os module
import os

# set cluster number
N_CLUSTERS = 6

# columns that characterize a player's performance profile for clustering
CLUSTER_STAT_COLS = [f'mean_{c}' for c in STAT_COLS] + [f'trend_{c}' for c in STAT_COLS]
cluster_feats = [c for c in CLUSTER_STAT_COLS if c in df.columns]

# -- 8a. fit clustering pupeline on the full dataset (no train/test split
#        because cluster labels are derived features, not learned target)
#        impute and scale before PCA + KMeans
cluster_imputer = SimpleImputer(strategy='median')
cluster_scaler  = StandardScaler()

X_cluster_raw    = df[cluster_feats].values
X_cluster_imp    = cluster_imputer.fit_transform(X_cluster_raw)
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster_imp)

# reduce to 20 principal components to balance variance & meaningful distance metric
pca_cluster   = PCA(n_components=20, random_state=42)
X_cluster_pca = pca_cluster.fit_transform(X_cluster_scaled)

# -- 8b. Elbow check: run K-Means for k=2...10, and store inertia
print("\nComputing K-Means elbow curve…")
inertias = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_pca)
    inertias[k] = km.inertia_

# -- 8c. fit the final K-Means with chosen n_clusters
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_cluster_pca)
df['cluster']  = cluster_labels

# -- 8d. distance to each cluster centroid for soft membership features
# hard labels discard information about how confidently a player belongs to a cluster
# A player equidistant from two centroids is different than a player who sits near one
centroid_dists = kmeans.transform(X_cluster_pca)
for k in range(N_CLUSTERS):
    df[f'dist_cluster_{k}'] = centroid_dists[:, k]

# -- 8e. print cluster AAV profile so we can view the archetypes
print("\nCluster profiles (MLB contracts only):")
print(
    df[df['mlb_contract'] == 1]
    .groupby('cluster')
    .agg(n=('aav','size'), median_aav=('aav','median'),
         median_war=('last_WAR','median'), median_wRC=('last_wRC+','median'))
    .sort_values('median_aav', ascending=False)
    .to_string()
)

# ── 8f. save cluster lookup so it can be applied at inference time ────────────
joblib.dump(
    (cluster_imputer, cluster_scaler, pca_cluster, kmeans, cluster_feats),
    '/content/v3_cluster_pipeline.pkl'
)



Computing K-Means elbow curve…

Cluster profiles (MLB contracts only):
           n  median_aav  median_war  median_wRC
cluster                                         
0        230  10000000.0    1.673767  114.150068
2         29   4000000.0    1.455521  114.305882
5         93   3250000.0    0.656598   86.289539
4        157   2000000.0    0.456019   88.279277
1         13   1400000.0   -0.283436   58.610735
3         21   1000000.0   -0.031964   49.889199


['/content/v3_cluster_pipeline.pkl']

## Part 9: Determine Features for Modeling


In [11]:
# Columns we never use as model inputs (identifiers, targets, raw cluster label
# is excluded here because we use the *distance* features instead — they are
# continuous and carry more information than a single integer label).

DROP_COLS = ['mlbam_id', 'name', 'contract_year', 'mlb_contract', 'aav', 'yrs']
feat_cols = [c for c in df.columns if c not in DROP_COLS]

X     = df[feat_cols]
y_cls = df['mlb_contract']

# Phase 2 uses only MLB-contract rows
mask_mlb = df['mlb_contract'] == 1
X_mlb    = df.loc[mask_mlb, feat_cols]
y_aav    = df.loc[mask_mlb, 'aav']
y_yrs    = df.loc[mask_mlb, 'yrs']

print(f"\nTotal features: {X.shape[1]}")



Total features: 230


## Part 10: Establish WAR Tiers

Why tier at these WAR values?

- < 0.5:  "fringe"   — replacement level; teams offer near-minimum or NRI
- 0.5-2:  "role"     — bench/roster piece; 1-year deals near league minimum
- 2-4:    "regular"  — starting-calibre; multi-year deals start appearing
- 4+:     "star"     — all-star quality; superstar money and long terms

These align with baseball convention and the inflection points visible in a WAR-vs-AAV scatter (the slope steepens sharply above 2 and again above 4).

Soft blending: within BLEND_RADIUS WAR of any boundary we interpolate between adjacent models using a sigmoid weight.  This eliminates the discontinuity that would occur if we used hard cutoffs alone.

In [12]:
# establish tiers
WAR_TIERS = {
    #  tier_name : (lower_bound_inclusive, upper_bound_exclusive)
    'fringe':  (float('-inf'), 0.5),
    'role':    (0.5,           2.0),
    'regular': (2.0,           4.0),
    'star':    (4.0,           float('inf')),
}
TIER_ORDER      = ['fringe', 'role', 'regular', 'star']
TIER_BOUNDARIES = [0.5, 2.0, 4.0]   # must match WAR_TIERS order
BLEND_RADIUS    = 0.5                # WAR units; blend within this distance of each boundary

# create function to assign WAR tier to players
def assign_war_tier(war_value: float) -> str:
    """Return the WAR tier name for a given WAR value."""
    for tier in TIER_ORDER:
        lo, hi = WAR_TIERS[tier]
        if lo <= war_value < hi:
            return tier
    return 'role'   # safe fallback

# create function to assign sigmoid weight
def sigmoid_weight(x: float, center: float, steepness: float = 4.0) -> float:
    """
    Sigmoid function returning a weight in (0, 1).
    At x == center → 0.5 (equal blend).
    At x >> center → 1.0 (fully upper tier).
    At x << center → 0.0 (fully lower tier).
    steepness controls how quickly the transition happens.
    """
    return 1.0 / (1.0 + np.exp(-steepness * (x - center)))


def get_tier_model(tier: str, n_samples: int):
    """
    Return an appropriate sklearn Pipeline for the given tier and sample size.

    Design rationale:
      - 'fringe' / 'role' tiers have small n and simpler salary structure
        → Ridge regression; robust when n < 50, less prone to overfitting
      - 'regular' / 'star' tiers have more data and richer non-linearities
        → GBM; captures the threshold effects and interaction terms
      - All tiers use imputation + scaling in the pipeline so they can be
        applied to single inference rows without separate preprocessing
    """
    if n_samples < 60 or tier == 'fringe':
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  StandardScaler()),
            ('reg',    Ridge(alpha=10.0))
        ])
    else:
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  StandardScaler()),
            ('reg',    GradientBoostingRegressor(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                min_samples_leaf=5,
                random_state=42
            ))
        ])


## Part 11: Tiered Model Training

In [13]:
# create function to fit tiered models for AAV and Years for each WAR tier
def fit_tiered_models(X_mlb_df, y_aav_s, y_yrs_s, war_col='last_WAR'):
    """
    Fit separate AAV and Years models for each WAR tier.

    Parameters
    ----------
    X_mlb_df : feature DataFrame (MLB contracts only)
    y_aav_s  : Series of AAV targets
    y_yrs_s  : Series of years targets
    war_col  : column in X_mlb_df holding last-season WAR

    Returns
    -------
    dict tier_name -> {'aav': fitted_pipeline, 'yrs': fitted_pipeline,
                       'n': int, 'war_range': str}
    """
    war_values = X_mlb_df[war_col].fillna(0.0)
    models     = {}

    for tier in TIER_ORDER:
        lo, hi = WAR_TIERS[tier]
        mask   = (war_values >= lo) & (war_values < hi)
        n      = mask.sum()
        print(f"\n  [{tier:8s}]  n={n:3d}  WAR [{lo:.1f}, {hi:.1f})")

        if n < 5:
            print(f"Too few samples — skipping tier (will use adjacent tier at inference)")
            models[tier] = None
            continue

        X_t   = X_mlb_df.loc[mask]
        ya_t  = y_aav_s.loc[mask]
        yy_t  = y_yrs_s.loc[mask]

        aav_m = get_tier_model(tier, n)
        yrs_m = get_tier_model(tier, n)

        # Log-scale AAV for all tiers — keeps loss function well-behaved
        aav_m.fit(X_t, np.log1p(ya_t))
        yrs_m.fit(X_t, yy_t)

        # Per-tier cross-validation (only meaningful for larger tiers)
        cv_folds = min(5, n // 10) if n >= 20 else 2
        if cv_folds >= 2:
            cv_aav = cross_val_score(aav_m, X_t, np.log1p(ya_t), cv=cv_folds, scoring='r2')
            cv_yrs = cross_val_score(yrs_m, X_t, yy_t,           cv=cv_folds, scoring='r2')
            print(f"    CV R² AAV : {cv_aav.mean():.3f} ± {cv_aav.std():.3f}")
            print(f"    CV R² Yrs : {cv_yrs.mean():.3f} ± {cv_yrs.std():.3f}")

        models[tier] = {
            'aav':       aav_m,
            'yrs':       yrs_m,
            'n':         int(n),
            'war_range': f"[{lo:.1f}, {hi:.1f})",
        }

    return models


def predict_tiered(war_value: float, row_df: pd.DataFrame,
                   tiered_models: dict) -> tuple[float, float]:
    """
    Predict AAV and years using the appropriate WAR-tier model, with
    sigmoid blending at boundaries.

    Parameters
    ----------
    war_value     : the player's last-season WAR
    row_df        : single-row DataFrame with all feature columns (aligned to feat_cols)
    tiered_models : dict from fit_tiered_models()

    Returns
    -------
    (pred_aav_dollars, pred_yrs_float)
    """
    def _predict_tier(tier):
        """Get (log_aav, yrs) prediction for a specific tier."""
        m = tiered_models.get(tier)
        if m is None:
            # Tier skipped (too few samples) — fall back to adjacent tier
            idx = TIER_ORDER.index(tier)
            for fallback in TIER_ORDER[max(0, idx-1):] + TIER_ORDER[:idx]:
                if tiered_models.get(fallback) is not None:
                    m = tiered_models[fallback]
                    break
        return (
            m['aav'].predict(row_df)[0],
            m['yrs'].predict(row_df)[0],
        )

    primary_tier = assign_war_tier(war_value)

    # Check if we are within blend range of any tier boundary
    for i, boundary in enumerate(TIER_BOUNDARIES):
        if abs(war_value - boundary) < BLEND_RADIUS:
            lower_tier = TIER_ORDER[i]
            upper_tier = TIER_ORDER[i + 1]

            log_aav_lo, yrs_lo = _predict_tier(lower_tier)
            log_aav_hi, yrs_hi = _predict_tier(upper_tier)

            # Weight toward upper tier as WAR increases past the boundary
            w_hi = sigmoid_weight(war_value, center=boundary)
            w_lo = 1.0 - w_hi

            blended_log_aav = w_lo * log_aav_lo + w_hi * log_aav_hi
            blended_yrs     = w_lo * yrs_lo     + w_hi * yrs_hi
            return np.expm1(blended_log_aav), blended_yrs

    # Not near any boundary — use primary tier directly
    log_aav, yrs = _predict_tier(primary_tier)
    return np.expm1(log_aav), yrs


## Part 12: Establish Feature Importance

In [14]:
# create function to show top features in pipeline
def top_features(pipe, feat_cols, n=15, label=''):
    """Extract and print top-n feature importances from a GBM or Ridge pipeline."""
    # estimator step is named 'clf' or 'reg' depending on the pipeline
    est = pipe.named_steps.get('reg') or pipe.named_steps.get('clf')
    if hasattr(est, 'feature_importances_'):
        imp = est.feature_importances_
    elif hasattr(est, 'coef_'):
        imp = np.abs(est.coef_)
    else:
        return pd.Series(dtype=float)

    fi = pd.Series(imp, index=feat_cols).nlargest(n)
    print(f"\nTop {n} features — {label}:")
    for name, v in fi.items():
        print(f"  {name:<42} {v:.4f}")
    return fi


## Part 13: Modeling Phase 1 - MLB vs Minor League Classifier

Model choice: Gradient Boosting (GBM)
- Handles missing values gracefully via imputation in the pipeline
- Captures non-linear interactions (e.g. WAR matters much more above 2.0)
- Built-in feature importance for explainability

We use StratifiedKFold to preserve the ~1:2 MLB:minor class ratio in splits.

In [15]:

print("\n" + "="*65)
print("PHASE 1 — MLB vs. Minor League Contract Classifier")
print("="*65)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

clf_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('clf',    GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=10, random_state=42
    ))
])

clf_pipe.fit(X_tr_c, y_tr_c)
y_pred_c = clf_pipe.predict(X_te_c)
y_prob_c = clf_pipe.predict_proba(X_te_c)[:, 1]

print("\nClassification Report:")
print(classification_report(y_te_c, y_pred_c, target_names=['Minor', 'MLB']))
auc_score = roc_auc_score(y_te_c, y_prob_c)
print(f"ROC-AUC: {auc_score:.4f}")

cv_auc = cross_val_score(
    clf_pipe, X, y_cls,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc'
)
print(f"5-Fold CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

fi_clf = top_features(clf_pipe, feat_cols, label='Classifier')



PHASE 1 — MLB vs. Minor League Contract Classifier

Classification Report:
              precision    recall  f1-score   support

       Minor       0.89      0.90      0.90       227
         MLB       0.79      0.78      0.78       109

    accuracy                           0.86       336
   macro avg       0.84      0.84      0.84       336
weighted avg       0.86      0.86      0.86       336

ROC-AUC: 0.9217
5-Fold CV AUC: 0.9131 ± 0.0127

Top 15 features — Classifier:
  mean_WAR                                   0.3208
  last_WAR                                   0.1152
  dist_cluster_1                             0.0265
  last_HR                                    0.0219
  delta2_WAR                                 0.0145
  mean_PA                                    0.0130
  trend_WAR                                  0.0104
  mean_wRC+                                  0.0100
  delta2_BsR                                 0.0100
  delta1_Def                                 0.0099

## Part 14: Phase 2 - Global Model Baseline for Tiered Comparison

We keep a single global AAV/years model.  This is the "baseline" against which tiered improvements are measured in the evaluation section and visualisation.

We model log(AAV + 1) rather than raw AAV because:
- Salary distributions are heavily right-skewed (a few $30M+ deals)
- Squared-error loss on raw dollars would over-weight megadeals
- Log scale aligns with how percentage raises actually work
- We exponentiate predictions back to dollars for reporting

Years is a small integer (mostly 1-6). We still use a continuous regressor and round at inference time. Compared to AAV, years is less right-skewed so we model it on the original scale.

In [16]:


print("\n" + "="*65)
print("PHASE 2 — GLOBAL BASELINE (for comparison with tiered models)")
print("="*65)

X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(
    X_mlb, y_aav, test_size=0.2, random_state=42
)
X_tr_y, X_te_y, y_tr_y, y_te_y = train_test_split(
    X_mlb, y_yrs, test_size=0.2, random_state=42
)

global_aav_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('reg',    GradientBoostingRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.04,
        subsample=0.8, min_samples_leaf=8, random_state=42
    ))
])
global_yrs_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('reg',    GradientBoostingRegressor(
        n_estimators=400, max_depth=3, learning_rate=0.04,
        subsample=0.8, min_samples_leaf=8, random_state=42
    ))
])

global_aav_pipe.fit(X_tr_a, np.log1p(y_tr_a))
global_yrs_pipe.fit(X_tr_y, y_tr_y)

global_pred_a = np.expm1(global_aav_pipe.predict(X_te_a))
global_pred_y = global_yrs_pipe.predict(X_te_y)

print(f"\nGlobal AAV  — R²: {r2_score(y_te_a, global_pred_a):.4f}  "
      f"MAE: ${mean_absolute_error(y_te_a, global_pred_a):,.0f}")
print(f"Global Years — R²: {r2_score(y_te_y, global_pred_y):.4f}  "
      f"MAE: {mean_absolute_error(y_te_y, global_pred_y):.3f} yrs")

fi_aav = top_features(global_aav_pipe, feat_cols, label='Global AAV Regressor')
fi_yrs = top_features(global_yrs_pipe, feat_cols, label='Global Years Regressor')



PHASE 2 — GLOBAL BASELINE (for comparison with tiered models)

Global AAV  — R²: 0.4756  MAE: $3,731,750
Global Years — R²: 0.5105  MAE: 0.599 yrs

Top 15 features — Global AAV Regressor:
  mean_BB%                                   0.1122
  delta2_Def                                 0.0543
  mean_WAR                                   0.0464
  mean_SLG                                   0.0455
  trend_G                                    0.0301
  mean_Spd                                   0.0283
  last_WAR                                   0.0263
  delta2_HardHit%                            0.0258
  trend_BsR                                  0.0238
  delta1_ISO                                 0.0231
  trend_Def                                  0.0229
  mean_ISO+                                  0.0215
  dist_cluster_1                             0.0198
  trend_BABIP                                0.0174
  dist_cluster_2                             0.0169

Top 15 features — Global Years

## Part 15: Phase 2 - Fit Tiered Models

In [17]:

print("\n" + "="*65)
print("PHASE 2 — TIERED WAR MODELS")
print("="*65)

tiered_models = fit_tiered_models(X_mlb, y_aav, y_yrs, war_col='last_WAR')



PHASE 2 — TIERED WAR MODELS

  [fringe  ]  n=190  WAR [-inf, 0.5)
    CV R² AAV : -4.020 ± 4.695
    CV R² Yrs : -0.227 ± 0.176

  [role    ]  n=233  WAR [0.5, 2.0)
    CV R² AAV : -0.945 ± 1.981
    CV R² Yrs : -0.452 ± 0.556

  [regular ]  n= 89  WAR [2.0, 4.0)
    CV R² AAV : -2.248 ± 2.524
    CV R² Yrs : -0.094 ± 0.265

  [star    ]  n= 31  WAR [4.0, inf)
    CV R² AAV : -0.563 ± 0.758
    CV R² Yrs : -0.482 ± 0.571


## Part 16: Tiered vs Global Model Comparison

Evaluate both the global model and the tiered model on the SAME test rows, broken down by tier.  This shows where a tiered model can help or hurt.

In [18]:


print("\n" + "="*65)
print("TIERED vs. GLOBAL EVALUATION (by WAR tier)")
print("="*65)

df_mlb = df[mask_mlb].copy()
df_mlb['last_WAR_filled'] = df_mlb['last_WAR'].fillna(0.0)
df_mlb['war_tier'] = df_mlb['last_WAR_filled'].apply(assign_war_tier)

tier_eval_rows = []
for tier in TIER_ORDER:
    tier_mask = df_mlb['war_tier'] == tier
    if tier_mask.sum() < 5:
        continue

    X_t   = df_mlb.loc[tier_mask, feat_cols]
    ya_t  = df_mlb.loc[tier_mask, 'aav'].values
    yy_t  = df_mlb.loc[tier_mask, 'yrs'].values

    # Global predictions
    g_aav = np.expm1(global_aav_pipe.predict(X_t))
    g_yrs = global_yrs_pipe.predict(X_t)

    # Tiered predictions (with blending)
    war_vals  = df_mlb.loc[tier_mask, 'last_WAR_filled'].values
    t_aav_arr = []
    t_yrs_arr = []
    for i, wv in enumerate(war_vals):
        row_df = X_t.iloc[[i]]
        pa, py = predict_tiered(wv, row_df, tiered_models)
        t_aav_arr.append(pa)
        t_yrs_arr.append(py)
    t_aav = np.array(t_aav_arr)
    t_yrs = np.array(t_yrs_arr)

    row = {
        'tier':           tier,
        'n':              tier_mask.sum(),
        'global_aav_r2':  r2_score(ya_t, g_aav),
        'tiered_aav_r2':  r2_score(ya_t, t_aav),
        'global_aav_mae': mean_absolute_error(ya_t, g_aav),
        'tiered_aav_mae': mean_absolute_error(ya_t, t_aav),
        'global_yrs_r2':  r2_score(yy_t, g_yrs),
        'tiered_yrs_r2':  r2_score(yy_t, t_yrs),
    }
    tier_eval_rows.append(row)
    print(
        f"\n  {tier.upper():8s} (n={tier_mask.sum():3d})\n"
        f"    AAV  R²  — global: {row['global_aav_r2']:+.3f}  tiered: {row['tiered_aav_r2']:+.3f}  "
        f"Δ={row['tiered_aav_r2'] - row['global_aav_r2']:+.3f}\n"
        f"    AAV MAE — global: ${row['global_aav_mae']:>10,.0f}  tiered: ${row['tiered_aav_mae']:>10,.0f}\n"
        f"    Yrs  R²  — global: {row['global_yrs_r2']:+.3f}  tiered: {row['tiered_yrs_r2']:+.3f}"
    )

tier_eval_df = pd.DataFrame(tier_eval_rows)




TIERED vs. GLOBAL EVALUATION (by WAR tier)

  FRINGE   (n=190)
    AAV  R²  — global: +0.817  tiered: +0.198  Δ=-0.619
    AAV MAE — global: $   647,289  tiered: $ 1,528,837
    Yrs  R²  — global: +0.443  tiered: +0.729

  ROLE     (n=233)
    AAV  R²  — global: +0.716  tiered: +0.628  Δ=-0.088
    AAV MAE — global: $ 1,388,089  tiered: $ 1,637,611
    Yrs  R²  — global: +0.734  tiered: +0.888

  REGULAR  (n= 89)
    AAV  R²  — global: +0.698  tiered: +0.940  Δ=+0.242
    AAV MAE — global: $ 2,980,497  tiered: $ 1,213,954
    Yrs  R²  — global: +0.674  tiered: +0.955

  STAR     (n= 31)
    AAV  R²  — global: +0.550  tiered: +0.851  Δ=+0.300
    AAV MAE — global: $ 5,578,000  tiered: $ 2,232,104
    Yrs  R²  — global: +0.937  tiered: +0.973


## Part 17: Build Inference Helper

Used to help us predict the contract terms for a baseball player entering free agency.

It will predict the likelihood of the player earning a MLB contract, and then the predicted AAV and years of the contract.

In [19]:

def predict_contract(mlbam_id: int, contract_year: int,
                     pos_override: str = None,
                     n_teams: int = None,
                     use_tiered: bool = True) -> tuple:
    """
    Full prediction for a player entering free agency.

    Steps
    -----
    1. Build performance + market features
    2. Assign cluster distances
    3. Optionally inject team-interest features (if known)
    4. Phase 1: P(MLB contract)
    5. Phase 2: predict AAV + years using tiered models (or global fallback)

    Parameters
    ----------
    mlbam_id      : MLBAM player ID
    contract_year : year the player is entering free agency
    pos_override  : position code (e.g. 'SS') to set position dummy
    n_teams       : override for n_teams_interested (if you have this info)
    use_tiered    : if False, use global model instead of tiered

    Returns
    -------
    (prob_mlb, pred_aav, pred_yrs)
    """
    # Build a minimal contract-row-like dict for build_player_features
    contract_row_proxy = {
        'mlbam_id':      mlbam_id,
        'Contract Year': contract_year,
    }
    # Fill team-interest cols with defaults (overridden below if n_teams given)
    for col in TEAM_INTEREST_COLS:
        contract_row_proxy[col] = 365 if col == 'first_rumor_doy' else 0
    contract_row_proxy = pd.Series(contract_row_proxy)

    feats = build_player_features(players, contract_row_proxy, MARKET_TABLE, aav_index)
    if feats is None:
        return f"No player data for MLBAM ID {mlbam_id}."

    # Optional: inject live team-interest count
    if n_teams is not None:
        feats['n_teams_interested'] = n_teams
        feats['rumor_intensity']    = n_teams  # simplified; no mention count

    # Cluster distances
    row_cluster = pd.DataFrame([feats])[cluster_feats]
    row_imp     = cluster_imputer.transform(row_cluster)
    row_scaled  = cluster_scaler.transform(row_imp)
    row_pca     = pca_cluster.transform(row_scaled)
    feats['cluster'] = int(kmeans.predict(row_pca)[0])
    for k in range(N_CLUSTERS):
        feats[f'dist_cluster_{k}'] = kmeans.transform(row_pca)[0][k]

    row = pd.DataFrame([feats])
    for pc in [c for c in feat_cols if c.startswith('pos_')]:
        row[pc] = 0.0
    if pos_override and f'pos_{pos_override}' in feat_cols:
        row[f'pos_{pos_override}'] = 1.0
    row = row.reindex(columns=feat_cols, fill_value=np.nan)

    prob_mlb = clf_pipe.predict_proba(row)[0][1]
    war_val  = feats.get('last_WAR', 0.0) or 0.0

    if use_tiered:
        pred_aav, pred_yrs = predict_tiered(war_val, row, tiered_models)
        model_label = f"Tiered ({assign_war_tier(war_val)} tier)"
    else:
        pred_aav = np.expm1(global_aav_pipe.predict(row)[0])
        pred_yrs = global_yrs_pipe.predict(row)[0]
        model_label = "Global"

    name_rows = players[players['MLBAMID'] == mlbam_id]
    name = name_rows['Name'].iloc[0] if len(name_rows) > 0 else str(mlbam_id)

    print(f"\n{'─'*58}")
    print(f"  Player       : {name}")
    print(f"  FA Year      : {contract_year}  |  WAR tier: {assign_war_tier(war_val)}")
    print(f"  Model        : {model_label}")
    print(f"  P(MLB deal)  : {prob_mlb:.1%}")
    print(f"  Pred AAV     : ${pred_aav:>12,.0f}")
    print(f"  Pred Years   : {pred_yrs:.1f}")
    print(f"{'─'*58}")
    return prob_mlb, pred_aav, pred_yrs


# ── Demo predictions ──────────────────────────────────────────────────────────
print("\n" + "="*65)
print("SAMPLE PREDICTIONS vs. ACTUAL")
print("="*65)

sample_ids = (
    df[df['mlb_contract'] == 1]
    .sample(6, random_state=7)[['mlbam_id', 'name', 'contract_year', 'aav', 'yrs', 'last_WAR']]
)
for _, r in sample_ids.iterrows():
    predict_contract(int(r['mlbam_id']), int(r['contract_year']))
    print(f"    Actual AAV  : ${r['aav']:>12,.0f}  "
          f"|  Actual Yrs: {r['yrs']}  |  WAR: {r['last_WAR']:.1f}")




SAMPLE PREDICTIONS vs. ACTUAL

──────────────────────────────────────────────────────────
  Player       : Edwin Rios
  FA Year      : 2022  |  WAR tier: fringe
  Model        : Tiered (fringe tier)
  P(MLB deal)  : 91.6%
  Pred AAV     : $     252,605
  Pred Years   : 0.9
──────────────────────────────────────────────────────────
    Actual AAV  : $   1,000,000  |  Actual Yrs: 1  |  WAR: 0.1

──────────────────────────────────────────────────────────
  Player       : Jason Castro
  FA Year      : 2020  |  WAR tier: fringe
  Model        : Tiered (fringe tier)
  P(MLB deal)  : 82.8%
  Pred AAV     : $   2,531,800
  Pred Years   : 1.2
──────────────────────────────────────────────────────────
    Actual AAV  : $   3,500,000  |  Actual Yrs: 2  |  WAR: 0.2

──────────────────────────────────────────────────────────
  Player       : Jose Iglesias
  FA Year      : 2021  |  WAR tier: role
  Model        : Tiered (role tier)
  P(MLB deal)  : 99.3%
  Pred AAV     : $   5,861,070
  Pred Years 

## Part 18: Data Visualizations

In [20]:
# set overall color tones
# color paletter set to dark mode
CLR = {
    'bg':      '#0d1117', 'panel':   '#161b22',
    'accent1': '#58a6ff', 'accent2': '#3fb950',
    'accent3': '#f78166', 'accent4': '#d2a8ff',
    'accent5': '#ffa657', 'text':    '#e6edf3',
    'muted':   '#8b949e', 'grid':    '#21262d',
}

# cluster color wheel
CLUSTER_COLORS = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#39d353']
TIER_COLORS    = {'fringe': '#8b949e', 'role': '#58a6ff',
                  'regular': '#3fb950', 'star': '#ffa657'}

plt.rcParams.update({
    'figure.facecolor': CLR['bg'],  'axes.facecolor': CLR['panel'],
    'text.color':       CLR['text'], 'axes.labelcolor': CLR['text'],
    'xtick.color':      CLR['text'], 'ytick.color': CLR['text'],
    'axes.edgecolor':   CLR['grid'], 'grid.color': CLR['grid'],
    'font.family':      'DejaVu Sans',
})

def ax_style(ax, title, xlabel='', ylabel='', fontsize=12):
    ax.set_facecolor(CLR['panel'])
    ax.set_title(title, color=CLR['text'], fontsize=fontsize, fontweight='bold', pad=10)
    if xlabel: ax.set_xlabel(xlabel, color=CLR['muted'], fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=CLR['muted'], fontsize=9)
    ax.grid(True, alpha=0.3, color=CLR['grid'])
    for sp in ax.spines.values(): sp.set_edgecolor(CLR['grid'])

def feat_importance_bar(ax, fi_series, title):
    fi = fi_series.sort_values()
    colors = []
    for n in fi.index:
        if any(ti in n for ti in TEAM_INTEREST_COLS):
            colors.append(CLR['accent5'])
        elif 'last_' in n:
            colors.append(CLR['accent1'])
        elif 'delta' in n or 'trend' in n:
            colors.append(CLR['accent2'])
        elif 'dist_cluster' in n or n == 'cluster':
            colors.append(CLR['accent4'])
        elif any(m in n for m in ['aav_market','cbt','revenue','contract_year_num']):
            colors.append(CLR['accent3'])
        else:
            colors.append(CLR['muted'])
    ax.barh(fi.index, fi.values, color=colors, edgecolor=CLR['bg'], height=0.7)
    ax_style(ax, title, 'Importance', fontsize=11)
    ax.tick_params(axis='y', labelsize=7)
    legend_els = [
        Patch(facecolor=CLR['accent5'], label='Team Interest'),
        Patch(facecolor=CLR['accent1'], label='Last-season stat'),
        Patch(facecolor=CLR['accent2'], label='Delta / Trend'),
        Patch(facecolor=CLR['accent4'], label='Cluster feature'),
        Patch(facecolor=CLR['accent3'], label='Market / Inflation'),
        Patch(facecolor=CLR['muted'],   label='Mean / Position / Other'),
    ]
    ax.legend(handles=legend_els, facecolor=CLR['panel'], edgecolor=CLR['grid'],
              labelcolor=CLR['text'], fontsize=7, loc='lower right')

# ── Figure layout: 6 rows × 3 cols ───────────────────────────────────────────
fig = plt.figure(figsize=(22, 36))
fig.patch.set_facecolor(CLR['bg'])
gs  = gridspec.GridSpec(6, 3, figure=fig, hspace=0.48, wspace=0.35)

# ── Row 0: Phase 1 classifier diagnostics ────────────────────────────────────
from sklearn.metrics import roc_curve

ax00 = fig.add_subplot(gs[0, 0])
bars = ax00.bar(['Minor League','MLB'],
                [(y_cls==0).sum(), (y_cls==1).sum()],
                color=[CLR['accent3'],CLR['accent1']], width=0.5)
for b, c in zip(bars, [(y_cls==0).sum(), (y_cls==1).sum()]):
    ax00.text(b.get_x()+b.get_width()/2, b.get_height()+15,
              f'{c:,}', ha='center', color=CLR['text'], fontsize=10, fontweight='bold')
ax_style(ax00, 'Contract Type Distribution', ylabel='Count')

ax01 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(y_te_c, y_prob_c)
ax01.plot(fpr, tpr, color=CLR['accent1'], lw=2, label=f'AUC = {auc_score:.3f}')
ax01.plot([0,1],[0,1], '--', color=CLR['muted'], lw=1)
ax01.fill_between(fpr, tpr, alpha=0.15, color=CLR['accent1'])
ax01.legend(facecolor=CLR['panel'], edgecolor=CLR['grid'], labelcolor=CLR['text'])
ax_style(ax01, 'Phase 1 — ROC Curve', 'False Positive Rate', 'True Positive Rate')

ax02 = fig.add_subplot(gs[0, 2])
cm = confusion_matrix(y_te_c, y_pred_c)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax02,
            xticklabels=['Minor','MLB'], yticklabels=['Minor','MLB'],
            cbar=False, linewidths=0.5)
ax02.set_xlabel('Predicted', color=CLR['muted'])
ax02.set_ylabel('Actual',    color=CLR['muted'])
ax02.set_title('Confusion Matrix', color=CLR['text'], fontsize=12, fontweight='bold')

# ── Row 1: Market index + WAR distribution by tier ───────────────────────────

ax10 = fig.add_subplot(gs[1, 0])
years_sorted = sorted(aav_index.index)
idx_vals     = [aav_index[y] for y in years_sorted]
ax10.plot(years_sorted, idx_vals, 'o-', color=CLR['accent2'], lw=2, markersize=7)
ax10.axhline(1.0, color=CLR['muted'], ls='--', lw=1, label='2019 baseline')
ax10.fill_between(years_sorted, 1.0, idx_vals, alpha=0.12, color=CLR['accent2'])
ax10.annotate('COVID\n(2020)', xy=(2020, aav_index.get(2020,1.0)),
              xytext=(2020.4, 0.65),
              arrowprops=dict(arrowstyle='->', color=CLR['muted']),
              color=CLR['muted'], fontsize=8)
ax10.legend(facecolor=CLR['panel'], edgecolor=CLR['grid'],
            labelcolor=CLR['text'], fontsize=8)
ax_style(ax10, 'League AAV Market Index', 'Year', 'Index (2019=1.0)')

ax11 = fig.add_subplot(gs[1, 1])
war_vals_plot = df_mlb['last_WAR'].dropna()
for tier in TIER_ORDER:
    lo, hi = WAR_TIERS[tier]
    subset = war_vals_plot[(war_vals_plot >= lo) & (war_vals_plot < hi)]
    ax11.hist(subset, bins=15, alpha=0.7, color=TIER_COLORS[tier],
              edgecolor=CLR['bg'], label=f'{tier} (n={len(subset)})')
for b in TIER_BOUNDARIES:
    ax11.axvline(b, color=CLR['muted'], ls='--', lw=1)
ax11.legend(facecolor=CLR['panel'], edgecolor=CLR['grid'],
            labelcolor=CLR['text'], fontsize=8)
ax_style(ax11, 'WAR Distribution by Tier (MLB contracts)', 'last_WAR', 'Count')

ax12 = fig.add_subplot(gs[1, 2])
# Team-interest distribution (shows neutral defaults if cache missing)
ti_vals = df['n_teams_interested'] if HAS_TEAM_INTEREST else pd.Series([0]*len(df))
ax12.hist(ti_vals, bins=20, color=CLR['accent5'], edgecolor=CLR['bg'], alpha=0.85)
ax_style(ax12, 'Teams Interested Distribution\n(0 = cache not loaded)',
         'n_teams_interested', 'Count')
status = "✓ Cache loaded" if HAS_TEAM_INTEREST else "⚠ Defaults (run scraper)"
ax12.text(0.97, 0.95, status, transform=ax12.transAxes,
          ha='right', va='top', color=CLR['accent2'] if HAS_TEAM_INTEREST else CLR['accent3'],
          fontsize=8)

# ── Row 2: Global model diagnostics ──────────────────────────────────────────

ax20 = fig.add_subplot(gs[2, 0])
ax20.scatter(y_te_a/1e6, global_pred_a/1e6, alpha=0.45, s=20,
             color=CLR['accent2'], edgecolors='none')
lim = max(y_te_a.max(), global_pred_a.max())/1e6 * 1.05
ax20.plot([0,lim],[0,lim],'--', color=CLR['muted'], lw=1)
ax20.set_xlim(0,lim); ax20.set_ylim(0,lim)
ax20.text(0.05,0.92,
          f'R²={r2_score(y_te_a, global_pred_a):.3f}\n'
          f'MAE=${mean_absolute_error(y_te_a, global_pred_a)/1e6:.1f}M',
          transform=ax20.transAxes, color=CLR['accent2'], fontsize=9)
ax_style(ax20, 'Global AAV: Actual vs. Predicted', 'Actual ($M)', 'Predicted ($M)')

ax21 = fig.add_subplot(gs[2, 1])
rng = np.random.RandomState(0)
jitter = rng.uniform(-0.15, 0.15, len(y_te_y))
ax21.scatter(y_te_y + jitter, global_pred_y, alpha=0.4, s=15,
             color=CLR['accent3'], edgecolors='none')
lim_y = max(y_te_y.max(), global_pred_y.max()) + 0.5
ax21.plot([0,lim_y],[0,lim_y], '--', color=CLR['muted'], lw=1)
ax21.text(0.05,0.92,
          f'R²={r2_score(y_te_y, global_pred_y):.3f}\n'
          f'MAE={mean_absolute_error(y_te_y, global_pred_y):.2f}yrs',
          transform=ax21.transAxes, color=CLR['accent3'], fontsize=9)
ax_style(ax21, 'Global Years: Actual vs. Predicted', 'Actual (yrs)', 'Predicted')

ax22 = fig.add_subplot(gs[2, 2])
ks = sorted(inertias.keys())
ax22.plot(ks, [inertias[k] for k in ks], 'o-', color=CLR['accent1'], lw=2, markersize=7)
ax22.axvline(N_CLUSTERS, color=CLR['accent3'], ls='--', lw=1.5,
             label=f'Chosen k={N_CLUSTERS}')
ax22.legend(facecolor=CLR['panel'], edgecolor=CLR['grid'],
            labelcolor=CLR['text'], fontsize=8)
ax_style(ax22, 'K-Means Elbow Curve', 'k', 'Inertia')

# ── Row 3: Tiered model comparison ───────────────────────────────────────────

ax30 = fig.add_subplot(gs[3, :2])
if not tier_eval_df.empty:
    x    = np.arange(len(tier_eval_df))
    w    = 0.35
    bars_g = ax30.bar(x - w/2, tier_eval_df['global_aav_r2'], w,
                      label='Global', color=CLR['muted'], edgecolor=CLR['bg'])
    bars_t = ax30.bar(x + w/2, tier_eval_df['tiered_aav_r2'], w,
                      label='Tiered', color=CLR['accent2'], edgecolor=CLR['bg'])
    ax30.set_xticks(x)
    ax30.set_xticklabels([f"{r['tier']}\n(n={r['n']})" for _, r in tier_eval_df.iterrows()])
    ax30.legend(facecolor=CLR['panel'], edgecolor=CLR['grid'], labelcolor=CLR['text'])
    ax30.axhline(0, color=CLR['muted'], lw=0.5)
ax_style(ax30, 'AAV R² — Global vs. Tiered Model (by WAR Tier)', ylabel='R²')

ax31 = fig.add_subplot(gs[3, 2])
if not tier_eval_df.empty:
    delta_r2 = tier_eval_df['tiered_aav_r2'] - tier_eval_df['global_aav_r2']
    colors_d = [CLR['accent2'] if v >= 0 else CLR['accent3'] for v in delta_r2]
    ax31.barh(tier_eval_df['tier'], delta_r2, color=colors_d, edgecolor=CLR['bg'], height=0.5)
    ax31.axvline(0, color=CLR['muted'], lw=1)
ax_style(ax31, 'Tiered AAV R² Improvement\nvs. Global', 'ΔR² (tiered − global)')

# ── Row 4: Cluster visualisation ─────────────────────────────────────────────

ax40 = fig.add_subplot(gs[4, :2])
pca_2d = PCA(n_components=2, random_state=42)
X_2d   = pca_2d.fit_transform(X_cluster_pca)
for k in range(N_CLUSTERS):
    mask_k = cluster_labels == k
    ax40.scatter(X_2d[mask_k,0], X_2d[mask_k,1],
                 c=CLUSTER_COLORS[k], alpha=0.45, s=12, edgecolors='none',
                 label=f'Cluster {k}')
ax40.legend(facecolor=CLR['panel'], edgecolor=CLR['grid'],
            labelcolor=CLR['text'], fontsize=8, ncol=2)
ax_style(ax40, 'Player Clusters in PCA Space', 'PC1', 'PC2')

ax41 = fig.add_subplot(gs[4, 2])
medians = (df[df['mlb_contract']==1].groupby('cluster')['aav']
           .median().sort_values(ascending=True))
ax41.barh([f'Cluster {i}' for i in medians.index], medians.values/1e6,
          color=[CLUSTER_COLORS[i] for i in medians.index],
          edgecolor=CLR['bg'], height=0.6)
ax_style(ax41, 'Median AAV by Cluster\n(MLB contracts)', 'Median AAV ($M)')

# ── Row 5: Feature importances ────────────────────────────────────────────────

ax50 = fig.add_subplot(gs[5, 0])
feat_importance_bar(ax50, fi_clf, 'Top 15 Features — Phase 1 Classifier')

ax51 = fig.add_subplot(gs[5, 1])
feat_importance_bar(ax51, fi_aav, 'Top 15 Features — Global AAV')

ax52 = fig.add_subplot(gs[5, 2])
feat_importance_bar(ax52, fi_yrs, 'Top 15 Features — Global Years')

fig.suptitle('MLB Free Agent Contract Prediction Pipeline  —  v3',
             fontsize=20, fontweight='bold', color=CLR['text'], y=0.997)

out_png = '/content/mlb_contract_model_v3_results.png'
plt.savefig(out_png, dpi=150, bbox_inches='tight', facecolor=CLR['bg'])
print(f"\n✓ Visualisation saved → {out_png}")



✓ Visualisation saved → /content/mlb_contract_model_v3_results.png


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# 19.  SAVE ALL ARTEFACTS
# ─────────────────────────────────────────────────────────────────────────────

joblib.dump(clf_pipe,        '/content/v3_phase1_classifier.pkl')
joblib.dump(global_aav_pipe, '/content/v3_global_aav_regressor.pkl')
joblib.dump(global_yrs_pipe, '/content/v3_global_yrs_regressor.pkl')
joblib.dump(tiered_models,   '/content/v3_tiered_models.pkl')
joblib.dump(feat_cols,       '/content/v3_feature_columns.pkl')
joblib.dump(tier_eval_df,    '/content/v3_tier_evaluation.pkl')

print("✓ Models saved.")
print("\nAll done.")

✓ Models saved.

All done.
